# Tutorial: Geometria em EMSim

Este notebook demonstra **formas básicas**, **operações** (subtracção) e **visualizações** com PyVista, e como **construir um waveguide** rectangular.

## 1. Formas básicas

As primitivas são: `Box`, `Cylinder`, `Sphere`. Todas têm `bounds()` e `to_pyvista()`.

In [ ]:
from emsim.geometry import Box, Cylinder, Sphere

# Caixa: x_min, x_max, y_min, y_max, z_min, z_max [m]
caixa = Box(0.0, 0.01, 0.0, 0.005, 0.0, 0.02)
print("Box bounds:", caixa.bounds())

# Cilindro: centro (x,y), raio, z_min, z_max (eixo ao longo de z)
cilindro = Cylinder(center_x=0.005, center_y=0.0025, radius=0.002, z_min=0.0, z_max=0.02)
print("Cylinder bounds:", cilindro.bounds())

# Esfera: centro (x,y,z), raio
esfera = Sphere(center_x=0.005, center_y=0.0025, center_z=0.01, radius=0.003)
print("Sphere bounds:", esfera.bounds())

## 2. Visualizações

Use `plot_geometry(geom)` para desenhar no Jupyter. Pode usar `backend="pyvista"` (interactivo) ou `backend="matplotlib"`. Para guardar: `save_path="figures/box.png"`.

In [ ]:
from emsim.geometry import plot_geometry

# Desenhar a caixa (no Jupyter aparece inline)
plot_geometry(caixa, backend="pyvista", notebook=True)

# Para guardar em ficheiro (descomente e ajuste o path):
# plot_geometry(caixa, backend="matplotlib", save_path="figures/box.png")

In [ ]:
# Criar pasta e guardar figuras
import os
os.makedirs("figures", exist_ok=True)
plot_geometry(cilindro, backend="matplotlib", save_path="figures/cylinder.png")
plot_geometry(esfera, backend="matplotlib", save_path="figures/sphere.png")
print("Figuras guardadas em figures/")

## 3. Operações booleanas (subtracção)

Podemos subtrair uma forma a outra usando PyVista: por exemplo, uma **caixa com um furo cilíndrico** (domínio menos cilindro).

In [ ]:
import pyvista as pv

# Caixa e cilindro como meshes PyVista
box_mesh = caixa.to_pyvista().triangulate()
cyl_mesh = cilindro.to_pyvista().triangulate()

# Subtracção booleana: caixa - cilindro = caixa com furo
caixa_com_furo = box_mesh.boolean_difference(cyl_mesh)

# Visualizar
pl = pv.Plotter(notebook=True)
pl.add_mesh(caixa_com_furo, show_edges=True, color="tan")
pl.show()

## 4. Construir um waveguide rectangular

O **RectangularWaveguide** representa um guia de ondas oco (paredes PEC). Parâmetros: largura `a`, altura `b`, comprimento `length` (em metros).

In [ ]:
from emsim.geometry import RectangularWaveguide

# Guia WR-42 (banda Ku): a ≈ 10.67 mm, b ≈ 4.32 mm
a = 10.67e-3   # largura (x) [m]
b = 4.32e-3    # altura (y) [m]
length = 50e-3 # comprimento (z) [m]

waveguide = RectangularWaveguide(a=a, b=b, length=length)

print("Waveguide:", waveguide)
print("x_range:", waveguide.x_range)
print("y_range:", waveguide.y_range)
print("z_range:", waveguide.z_range)
print("bounds:", waveguide.bounds())

In [ ]:
# Visualizar o waveguide
plot_geometry(waveguide, backend="pyvista", notebook=True)

### Usar o waveguide na simulação

O mesmo objecto `RectangularWaveguide` é usado pela `Simulation` quando o tipo de geometria no YAML é `rectangular_waveguide`. Exemplo de config:

In [ ]:
config = {
    "geometry": {
        "type": "rectangular_waveguide",
        "a": a,
        "b": b,
        "length": length,
    },
    "frequency": {"f0": 10e9},
    "grid": {"resolution": 20, "courant": 0.5},
    "boundaries": {"pml_faces": ["z-", "z+"]},
    "run": {"n_steps": 500},
}

# Criar simulação a partir do config (requer load_yaml ou from_config)
# from emsim.simulation import Simulation
# sim = Simulation.from_config(config)
# sim.build()
# sim.run()

print("Config de exemplo para rectangular_waveguide:")
for k, v in config["geometry"].items():
    print(f"  {k}: {v}")